### Genetic algorithm


In [3]:
import random
import numpy as np
import pandas as pd

from sklearn_extra.cluster import KMedoids
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

In [6]:
df = pd.read_csv('cleaned_data.csv')

numerical_cols = [
    'person_age',
    'person_income',
    'person_emp_length',
    'loan_amnt',
    'loan_int_rate',
    'loan_percent_income',
    'cb_person_cred_hist_length'
]

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

X = df.drop('loan_status', axis=1)
y = df['loan_status']
feature_names = list(X.columns)

X_sample = X.sample(n=5000, random_state=42)

print("Features:", feature_names)
print("Total:", len(feature_names))


population_size = 10
mutation_probability = 0.2
generations = 10

fitness_cache = {}

Features: ['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'person_home_ownership_OTHER', 'person_home_ownership_OWN', 'person_home_ownership_RENT', 'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE', 'loan_grade_B', 'loan_grade_C', 'loan_grade_D', 'loan_grade_E', 'loan_grade_F', 'loan_grade_G', 'cb_person_default_on_file_Y']
Total: 22


In [7]:
df.head()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,...,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE,loan_grade_B,loan_grade_C,loan_grade_D,loan_grade_E,loan_grade_F,loan_grade_G,cb_person_default_on_file_Y
0,-1.092709,-1.065582,0.086322,-1.355767,0.037262,0,-0.656393,-0.944161,False,True,...,False,False,False,True,False,False,False,False,False,False
1,-0.449204,-1.065582,-0.937026,-0.644230,0.574790,1,3.744111,-0.698676,False,False,...,True,False,False,False,True,False,False,False,False,False
2,-0.770957,-0.006651,-0.169515,4.020295,1.308065,1,3.369600,-0.944161,False,False,...,True,False,False,False,True,False,False,False,False,False
3,-0.610081,-0.216922,0.853833,4.020295,1.009783,1,3.556855,-0.453191,False,False,...,True,False,False,False,True,False,False,False,False,True
4,-1.092709,-1.059899,-0.681189,-1.118588,-1.205578,1,0.748023,-0.944161,False,True,...,False,False,True,False,False,False,False,False,False,False


In [8]:
# Generate a random individual
def create_individual():
    return [random.randint(0, 1) for _ in range(len(feature_names))]

In [9]:
def fitness(individual):

    key = tuple(individual)
    if key in fitness_cache:
        return fitness_cache[key]

    selected = [i for i in range(len(individual)) if individual[i] == 1]

    # ✅ FIX 1: too few features → reject
    if len(selected) < 3:
        return 0

    # ❌ optional safety: too many features (prevents noise explosion)
    if len(selected) > 10:
        return 0

    data = X_sample.iloc[:, selected].values

    model = KMedoids(
        n_clusters=3,
        init='k-medoids++',   
        metric='euclidean',
        random_state=42
    )

    try:
        labels = model.fit_predict(data)
    except:
        return 0

    # ✅ FIX 2: check cluster validity
    if len(set(labels)) < 3:
        return 0

    # ---- Silhouette ----
    try:
        sil_score = silhouette_score(data, labels)
    except:
        return 0

    # ---- Risk separation ----
    df_temp = X_sample.copy()
    df_temp['cluster'] = labels
    df_temp['loan_status'] = y.reindex(df_temp.index)

    default_rates = df_temp.groupby('cluster')['loan_status'].mean()

    if len(default_rates) < 2:
        risk_sep = 0
    else:
        risk_sep = default_rates.max() - default_rates.min()

    # normalize silhouette
    sil_score = (sil_score + 1) / 2

    score = 0.6 * sil_score + 0.4 * risk_sep

    fitness_cache[key] = score
    return score

In [10]:
# Select parents using tournament selection
def select_parent(population):
    tournament = random.sample(population, 3)
    return max(tournament, key=lambda x: fitness(x))

In [11]:
# Single-point crossover
def crossover(parent1, parent2):
    point = random.randint(1, len(parent1) - 1)
    return parent1[:point] + parent2[point:]


In [12]:
# Mutation
def mutate(individual):
    individual = individual.copy()
    for i in range(len(individual)):
        if random.random() < mutation_probability:
            individual[i] = 1 - individual[i]
    return individual

In [13]:
def genetic_algorithm():

    population = [create_individual() for _ in range(population_size)]

    for gen in range(generations):

        population = sorted(population, key=lambda x: fitness(x), reverse=True)

        best = population[0]
        best_fit = fitness(best)

        selected_features = [
            feature_names[i] for i in range(len(best)) if best[i] == 1
        ]

        print(f"\nGeneration {gen+1}")
        print("Selected Features:", selected_features)
        print(f"Fitness (Silhouette + Risk): {best_fit:.4f}")

        new_population = []

        for _ in range(population_size):
            p1 = select_parent(population)
            p2 = select_parent(population)

            child = crossover(p1, p2)
            child = mutate(child)

            new_population.append(child)

        population = new_population

    best = max(population, key=lambda x: fitness(x))
    best_fit = fitness(best)

    selected_features = [
        feature_names[i] for i in range(len(best)) if best[i] == 1
    ]

    print("\nFINAL RESULT")
    print("Best Features:", selected_features)
    print("Number of Features:", sum(best))
    print("Final Fitness:", best_fit)

    return best, best_fit

# =========================
# BASELINE
# =========================
baseline = [1] * len(feature_names)
baseline_score = fitness(baseline)

print("\nBaseline Score:", baseline_score)

# =========================
# RUN GA
# =========================
best_solution, best_score = genetic_algorithm()

print("\n--- COMPARISON ---")
print("Baseline:", baseline_score)
print("GA:", best_score)
print("Improvement:", best_score - baseline_score)
print("Features reduced:", len(feature_names), "→", sum(best_solution))


Baseline Score: 0

Generation 1
Selected Features: ['person_emp_length', 'loan_int_rate', 'loan_percent_income', 'loan_intent_EDUCATION', 'loan_intent_MEDICAL', 'loan_grade_F', 'loan_grade_G', 'cb_person_default_on_file_Y']
Fitness (Silhouette + Risk): 0.5059

Generation 2
Selected Features: ['person_income', 'person_emp_length', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'loan_intent_EDUCATION', 'loan_intent_MEDICAL', 'loan_grade_G']
Fitness (Silhouette + Risk): 0.4988

Generation 3
Selected Features: ['person_income', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT', 'loan_intent_PERSONAL', 'loan_grade_G']
Fitness (Silhouette + Risk): 0.5197

Generation 4
Selected Features: ['person_income', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'loan_intent_HOMEIMPROVEMENT', 'loan_intent_PERSONAL', 'loan_grade_F', 'loan_grade_G']
Fitness (Silhouette + Risk): 0.5071

Generation 5
Selected Features: [

In [15]:
final_data = df[['loan_percent_income', 'loan_intent_HOMEIMPROVEMENT', 'loan_grade_B', 'loan_grade_C', 'loan_grade_E', 'cb_person_default_on_file_Y']]
final_data.to_csv("ga_selected_features.csv", index=False)